# IA Generativa con Neo4j
Implementación de un pipeline con grafos y embeddings para recuperación semántica.

In [ ]:
from py2neo import Graph, Node, Relationship

# Conexión a Neo4j
graph = Graph("bolt://localhost:7687", auth=("neo4j", "password"))
print(graph.run("RETURN 1").data())

In [ ]:
# Crear nodos de usuarios y mensajes
user = Node("User", name="Alice")
msg1 = Node("Message", text="¿Qué es una red neuronal?")
msg2 = Node("Message", text="Explica el aprendizaje por refuerzo")

graph.create(user | msg1 | msg2)
graph.create(Relationship(user, "CREÓ", msg1))
graph.create(Relationship(user, "CREÓ", msg2))

In [ ]:
# Generar embeddings
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

embedding1 = model.encode(msg1["text"]).tolist()
embedding2 = model.encode(msg2["text"]).tolist()

msg1["embedding"] = embedding1
msg2["embedding"] = embedding2
graph.push(msg1)
graph.push(msg2)

In [ ]:
# Recuperar mensajes por similitud
import numpy as np

def cos_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

query = model.encode("¿Qué es el aprendizaje automático?")

for m in graph.nodes.match("Message"):
    sim = cos_sim(query, m["embedding"])
    print(f"Similitud con '{m['text']}': {round(sim, 2)}")